In [6]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.ai.topic_model import get_topic_model

print("Gerçek Türkçe kullanıcı yorumları indiriliyor...")

# Doğrudan çalışan canlı Türkçe yorum veri setleri
urls = [
    # 1. Garanti Açık Kaynak Türkçe Yorum Veri Seti (TTC3600 / Sentiment)
    "https://raw.githubusercontent.com/fatihakyol/turkish-text-classification/master/data/5000-text-cases.csv",
    # 2. Alternatif Ürün Yorumları Linki
    "https://raw.githubusercontent.com/yusufkaradag/turkish-sentiment-analysis/main/data/turkish_tokens.csv"
]

texts = []
for url in urls:
    try:
        # Metin sütununu otomatik yakalayalım
        df = pd.read_csv(url, encoding="utf-8", on_bad_lines="skip")
        
        # Sütun adları değişiklik gösterebileceği için metin içeren sütunu buluyoruz
        text_col = [col for col in df.columns if any(keyword in col.lower() for keyword in ["text", "comment", "review", "metin", "yorum"])][0]
        
        texts = df[text_col].dropna().astype(str).tolist()[:1000]
        print(f"✅ Veri başarıyla çekildi! Toplam {len(texts)} adet gerçek yorum yüklendi.")
        break
    except Exception as e:
        continue

if not texts:
    raise RuntimeError("Veri indirme başarısız oldu, bağlantıları kontrol edin.")

# 2. BERTopic Modelini Çalıştırma
print("\nBERTopic modeli çalıştırılıyor...")
topic_m = get_topic_model()
results = topic_m.fit_transform_topics(texts, nr_topics=6)

# 3. DoD Doğrulama ve Sonuçları Raporlama
print("\n" + "=" * 50)
print(f"SONUÇ: Toplam {len(results)} Tema Çıkarıldı")
print("=" * 50 + "\n")

if len(results) >= 5:
    print("✅ DoD Başarılı: En az 5 anlamlı tema oluşturuldu!\n")
else:
    print("⚠️ DoD Uyarısı: Tema sayısı 5'in altında kaldı.\n")

for topic in results:
    topic_id = topic["topic_id"]
    topic_name = topic["topic_name"]
    doc_count = topic["document_count"]
    keywords = [k["word"] for k in topic["keywords"][:5]]

    print(f"📌 Tema ID {topic_id}: {topic_name}")
    print(f"   Metin Sayısı: {doc_count}")
    print(f"   Kelime Bulutu Anahtar Kelimeleri: {', '.join(keywords)}")
    print("-" * 50)

Gerçek Türkçe kullanıcı yorumları indiriliyor...


RuntimeError: Veri indirme başarısız oldu, bağlantıları kontrol edin.

In [9]:
import pandas as pd

df = pd.read_csv("../data/comments.csv")
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440679 entries, 0 to 440678
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   text     440679 non-null  object
 1   label    440679 non-null  object
 2   dataset  440679 non-null  object
dtypes: object(3)
memory usage: 10.1+ MB


In [11]:
print(df.isnull().sum())
df["dataset"].value_counts()
df["dataset"].unique()


text       0
label      0
dataset    0
dtype: int64


array(['urun_yorumlari', 'wiki', 'HUMIR', 'tweet-pn', 'magaza_yorumlari',
       'random'], dtype=object)

In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.ai.topic_model import get_topic_model
from src.ai.pipeline import on_isle

# 1. Excel dosyasını oku
df = pd.read_excel("../data/ATC.xlsx")

# 2. Sadece normal yorumları seç (0 = normal)
normal_df = df[df["label"] == 0].copy()

# 3. Eksik yorumları sil
normal_df = normal_df.dropna(subset=["message"])

# 4. Ön İşleme Pipeline'ı (A2.1) Uygulama
processed_texts = []

for raw_msg in normal_df["message"]:
    raw_str = str(raw_msg).strip()
    if len(raw_str) < 5:
        continue
    
    # A2.1 Ön işleme: Mention/Hashtag temizliği, harf sadeleştirme, emoji normalizasyonu
    processed_obj = on_isle(raw_str)
    clean_txt = processed_obj.temiz_metin
    
    # Çok kısa veya boş kalmış metinleri filtreleme (En az 2 kelime)
    if clean_txt and len(clean_txt.split()) >= 2:
        processed_texts.append(clean_txt)


# Tekrarlayan metinleri temizleme
processed_texts = list(set(processed_texts))
print(f"Ön işleme ve filtreleme sonrası hazır yorum sayısı: {len(processed_texts)}")

# 5. Analiz için 2000 adetlik örneklem seçme
sample_texts = pd.Series(processed_texts).sample(
    n=min(2000, len(processed_texts)), 
    random_state=42
).tolist()

print(f"BERTopic için kullanılacak veri sayısı: {len(sample_texts)}")

Ön işleme ve filtreleme sonrası hazır yorum sayısı: 18669
BERTopic için kullanılacak veri sayısı: 2000


In [2]:
# 6. BERTopic Modelini Çalıştırma
topic_model = get_topic_model()

results = topic_model.fit_transform_topics(
    sample_texts,
    nr_topics=6
)

[topic_model] 2000 metin | min_topic_size=100 | n_neighbors=10
[topic_model] Ham (zorla birleştirmeden önceki) tema sayısı: 2


In [4]:
import json
from src.ai.topic_model import get_topic_model, prepare_text_for_topics

with open("../src/ai/data/mock_comments.json", encoding="utf-8") as f:
    mock_data = json.load(f)

texts = [prepare_text_for_topics(c["text"]) for c in mock_data]
texts = [t for t in texts if len(t.split()) >= 2]  # çok kısa olanları ele

topic_model = get_topic_model()
results = topic_model.fit_transform_topics(texts, nr_topics=6)

[topic_model] 2434 metin | min_topic_size=121 | n_neighbors=10
[topic_model] Ham (zorla birleştirmeden önceki) tema sayısı: 3


In [5]:
# 6. DoD Doğrulama ve Sonuçları Raporlama
print("\n" + "=" * 50)
print(f"SONUÇ: Toplam {len(results)} Tema Çıkarıldı")
print("=" * 50 + "\n")

if len(results) >= 5:
    print("✅ DoD Başarılı: En az 5 anlamlı tema oluşturuldu!\n")
else:
    print("⚠️ DoD Uyarısı: Tema sayısı 5'in altında kaldı.\n")

for topic in results:
    topic_id = topic["topic_id"]
    topic_name = topic["topic_name"]
    doc_count = topic["document_count"]
    keywords = [k["word"] for k in topic["keywords"][:5]]

    print(f"📌 Tema ID {topic_id}: {topic_name}")
    print(f"   Metin Sayısı: {doc_count}")
    print(f"   Kelime Bulutu Anahtar Kelimeleri: {', '.join(keywords)}")
    print("-" * 50)


SONUÇ: Toplam 3 Tema Çıkarıldı

⚠️ DoD Uyarısı: Tema sayısı 5'in altında kaldı.

📌 Tema ID 0: Video / paylaşım / içerik
   Metin Sayısı: 569
   Kelime Bulutu Anahtar Kelimeleri: video, paylaşım, içerik, rota, vlog
--------------------------------------------------
📌 Tema ID 1: Beğenmedim / kısmı / özellikle
   Metin Sayısı: 198
   Kelime Bulutu Anahtar Kelimeleri: beğenmedim, kısmı, özellikle, beğendim, beğendim özellikle
--------------------------------------------------
📌 Tema ID 2: Mısınız / tam / kombinin
   Metin Sayısı: 166
   Kelime Bulutu Anahtar Kelimeleri: mısınız, tam, kombinin, sporun, tarifin
--------------------------------------------------
